### Boosting

Sequential ensembles: each new learner corrects what previous ones got wrong, unlike bagging's parallel independent trees (see `bagging.ipynb`). Covers, in the order each one improves on the last: AdaBoost (reweighting, the historical origin), Classic GBM (gradient fitting, first-order), XGBoost (second-order, regularized, from-scratch build), LightGBM (leaf-wise, histogram, GOSS), CatBoost (ordered target statistics, native categoricals).

## Boosting: AdaBoost

The historical origin of boosting, predates gradient boosting entirely. Mechanically different from every gradient-boosted method below: it does not fit trees to gradients, it reweights training EXAMPLES so each new weak learner focuses on what previous ones got wrong, then combines all learners via a weighted vote.

#### 0. Core idea

Loop: fit a weak learner (often a decision stump, a 1-split tree) on the current sample weights, measure its weighted error, give it a voting weight based on how good it was, then boost the weights of the examples it got wrong so the next learner focuses on them. Final prediction: weighted vote across all learners, sign(sum(alpha_m * h_m(x))).

Toy setup, labels in {-1,+1} (AdaBoost's convention, not {0,1}), one feature mentions_IRS, with one genuinely ambiguous case to force a real weak-learner error:
```
doc1: IRS=1, y=+1 (gov)
doc2: IRS=1, y=+1 (gov)
doc3: IRS=1, y=-1 (romance, mentions IRS in passing, not actually impersonation)
doc4: IRS=0, y=-1 (romance)
```

#### 1. Round 1: fit, weigh, reweight

Initial weights: all equal, w_i = 1/4 = 0.25.

A stump splitting on IRS=1 vs IRS=0 predicts by majority vote within each side:
```
IRS=1 side: {doc1(+1), doc2(+1), doc3(-1)} -> majority = +1
IRS=0 side: {doc4(-1)} -> majority = -1

predictions: doc1=+1(correct), doc2=+1(correct), doc3=+1(WRONG, actual -1), doc4=-1(correct)
```
Weighted error rate: err = sum(w_i for misclassified) / sum(all w_i) = 0.25 / 1.0 = 0.25

Learner's voting weight (alpha): alpha = 0.5 * ln((1-err)/err) = 0.5 * ln(3) = 0.5493. Lower error gives higher alpha, a learner that is almost always right gets almost all the say in the final vote; err=0.5 (no better than random) gives alpha=0, that learner contributes nothing.

Reweight: w_i *= exp(-alpha * y_i * pred_i), then renormalize to sum to 1.
```
correctly classified (y*pred=+1): multiply by exp(-0.5493) = 0.5774
misclassified (y*pred=-1):        multiply by exp(+0.5493) = 1.7321

doc1: 0.25*0.5774 = 0.1443
doc2: 0.25*0.5774 = 0.1443
doc3: 0.25*1.7321 = 0.4330   <- the one it got wrong
doc4: 0.25*0.5774 = 0.1443

sum = 0.8660, normalize (divide each by 0.8660):
doc1=0.1667, doc2=0.1667, doc3=0.5000, doc4=0.1667
```
doc3's weight DOUBLED (0.25 to 0.50), the other three dropped. Round 2's weak learner trains on THIS reweighted data, so it is effectively forced to pay much more attention to doc3, the example round 1 got wrong. This is AdaBoost's version of "correcting the previous mistake", reweighting examples, not fitting a new model to a gradient the way every other method in this notebook does.

In [ ]:
import numpy as np

X = np.array([1, 1, 1, 0])       # mentions_IRS
y = np.array([1, 1, -1, -1])     # +1=gov, -1=romance
weights = np.full(4, 0.25)

# stump: IRS=1 -> majority vote of that side, IRS=0 -> majority vote of that side
preds = np.where(X == 1, 1, -1)  # matches the majority-vote stump above
print("predictions:", preds)

misclassified = preds != y
err = np.sum(weights[misclassified]) / np.sum(weights)
alpha = 0.5 * np.log((1 - err) / err)
print("weighted error:", err, "| alpha:", alpha)

weights = weights * np.exp(-alpha * y * preds)
weights = weights / weights.sum()
print("updated weights:", weights.round(4))

#### 2. Final prediction: weighted vote, not averaging

After M rounds, final prediction = sign(sum over m of alpha_m * h_m(x)). Each learner's vote is scaled by its own alpha, a learner that was very accurate (low error, high alpha) gets more say than a learner that barely beat random guessing (high error, low alpha). Different from Random Forest's equal-vote majority (every tree counts the same) and different from gradient boosting's raw score summation (no explicit per-learner confidence weight, the tree's own leaf values already encode that).

#### 3. Practical notes

Sensitive to noisy labels and outliers, a genuinely mislabeled example keeps getting misclassified round after round, its weight keeps compounding upward, eventually dominating training and distorting later learners. Gradient boosting methods (XGBoost, LightGBM, CatBoost, all below) are generally more robust to this since they fit smooth gradients rather than hard reweighting, and have built-in regularization AdaBoost lacks. AdaBoost is rarely the first choice in production now, gradient boosting variants dominate, but the reweighting mechanism itself reappears in other algorithms and is a common interview topic specifically because it is a genuinely different combination strategy than everything that came after it.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

X_2d = X.reshape(-1, 1)
ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=3, random_state=42)
ada.fit(X_2d, y)

print("predictions:", ada.predict(X_2d))
print("estimator weights (alpha per round):", ada.estimator_weights_)

## Boosting: Classic GBM

#### 0. Core idea

XGBoost's direct predecessor. Same sequential boosting loop, fewer optimizations. Same toy setup as the XGBoost note: doc1 (mentions_IRS=1, y=1=gov), doc2 (mentions_IRS=0, y=0=romance).

Loop: F=0, then repeatedly: compute gradient of the loss, fit a tree to that gradient, add the tree's scaled output to F.

Start: F=0 for both docs -> p = sigmoid(0) = 0.5 for both.
Gradient: g = p - y -> g1 = 0.5-1 = -0.5 (doc1), g2 = 0.5-0 = 0.5 (doc2).
Hessian: h = p(1-p) = 0.25 for both (same as the XGBoost note, since the math up to this point is identical).


#### 1. First order only, no Hessian in the leaf formula

XGBoost's leaf weight: w = -G/(H+lambda). Classic GBM's: w = -G/H. Drop lambda entirely, no regularization term.

Worked comparison, same two leaves as the XGBoost note (left=doc2, right=doc1):
```
XGBoost (lambda=1):    w_left = -0.5/(0.25+1) = -0.4,  w_right = 0.5/(0.25+1) = 0.4
Classic GBM (no lambda): w_left = -0.5/0.25 = -2.0,     w_right = 0.5/0.25 = 2.0
```
Same gradients, same data. Classic GBM's leaf values are 5x more extreme. That's lambda's job made concrete: it pulls leaf updates toward zero, classic GBM has no such pull.


In [ ]:
g_doc1, g_doc2 = -0.5, 0.5
h_doc1, h_doc2 = 0.25, 0.25
lam = 1

xgb_w_right = -g_doc1 / (h_doc1 + lam)
xgb_w_left = -g_doc2 / (h_doc2 + lam)
gbm_w_right = -g_doc1 / h_doc1
gbm_w_left = -g_doc2 / h_doc2

print(f"XGBoost leaf weights: left={xgb_w_left}, right={xgb_w_right}")
print(f"Classic GBM leaf weights: left={gbm_w_left}, right={gbm_w_right}")

#### 2. No gamma either, splits happen more readily

No min-gain-to-split term. Overfitting control relies only on learning_rate, max_depth, subsample. Fewer levers than XGBoost's L1/L2/gamma/min_child_weight combination.

#### 3. Slower: no histogram optimization

No "hist" tree_method equivalent by default, no parallel split-finding within a single tree's construction. Fine on small data, scales worse than XGBoost/LightGBM on large sparse matrices like TF-IDF.

#### 4. Practical consequence of the extreme leaf values

The update F += eta*w would swing wildly at eta=0.3 with w=+-2.0 (classic GBM), versus a much gentler swing with w=+-0.4 (XGBoost). This is exactly why sklearn's GradientBoostingClassifier defaults to a smaller learning_rate (0.1) than XGBoost's default (0.3), it has to compensate for having no lambda to dampen the leaf values itself.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np

X = np.array([[1], [0]])  # mentions_IRS only
y = [1, 0]  # 1=gov, 0=romance

gbm = GradientBoostingClassifier(n_estimators=10, learning_rate=0.1, max_depth=1, random_state=42)
gbm.fit(X, y)

print("predicted probabilities:", gbm.predict_proba(X))
print("predictions:", gbm.predict(X))

# inspect the first tree's structure to see the split it found
first_tree = gbm.estimators_[0, 0]
print("\nfirst tree split feature index:", first_tree.tree_.feature[0])
print("first tree split threshold:", first_tree.tree_.threshold[0])

## Boosting: XGBoost

### Machine Learning Fundamentals, XGBoost

Builds on: [03-log-regr.ipynb](log-regr.ipynb) (g=p-y gradient), [04-decision-tree.ipynb](decision-tree.ipynb) (split-search structure, Gini replaced by Gain here).

Plain terms: instead of one big model, build many small trees one after another, each one correcting the mistakes left over from everything built before it. Each tree looks at where the current predictions are wrong and how confident those wrong predictions were, then makes a small, cautious adjustment in that direction. Enough small corrections stacked together add up to a strong model.

1. Core Intuition and Boosting Assembly: XGBoost builds an ensemble step-by-step: $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$. It starts with a base guess $F_0$ (like average house price or base log-odds $0.5$). Each new decision tree $h_m(x)$ predicts remaining errors (residuals) of the previous step. The learning rate $\eta$ scales down each tree's contribution to prevent overshooting.

2. Simple Residuals ($g_i$) and Weights ($h_i$): At each step, every data point $i$ calculates two values based on its current error:
* **Gradient ($g_i$):** Direction of error ($p_i - y_i$ for classification, $\hat{y}_i - y_i$ for regression). Negative gradient $-g_i$ is literally the residual error.
* **Hessian ($h_i$):** Confidence weight ($1$ for regression, $p_i(1 - p_i)$ for classification).

3. The Only Two Core Formulas: Instead of standard Gini or MSE, XGBoost uses $G = \sum g_i$ (total error in a node) and $H = \sum h_i$ (total confidence in a node):
* **Leaf Output Value ($w^*$):** $w^* = -\frac{G}{H + \lambda}$ (where $\lambda$ is L2 regularization that shrinks large outputs).
* **Node Similarity Score ($SS$):** $SS = \frac{G^2}{2(H + \lambda)}$ (measures how well the node isolates remaining error).

4. Evaluating Splits with Gain: To test if a feature split is worth making, XGBoost compares child node scores to the parent score:

$$\text{Gain} = \frac{1}{2}\left[ SS_L + SS_R - SS_{\text{Parent}} \right] - \gamma$$

If $\text{Gain} > 0$, the split is accepted. The hyperparameter $\gamma$ sets the minimum threshold to prevent useless splits.

5. Node Cover & Min Child Weight: Cover is simply the sum of Hessians in a node ($\text{Cover} = H = \sum h_i$). The parameter `min_child_weight` stops a split if either child ends up with $H < \text{min\_child\_weight}$. In regression, Cover is just row count; in classification, it is sample size weighted by uncertainty $p(1-p)$.

7. Bottom-Up Tree Pruning: Trees grow top-down to `max_depth`. Then XGBoost checks branches bottom-up: if $\text{Gain} < 0$ at a node, the split is pruned, and child leaves collapse back into a single leaf with weight $w^* = -\frac{G_{\text{Parent}}}{H_{\text{Parent}} + \lambda}$.

8. Missing Value Routing (Sparsity Awareness): If data points have missing values ($N/A$) for a feature, XGBoost tests sending all missing points to the Left branch, then to the Right branch. The path that produces the higher Gain is saved as the node's permanent default path for missing values.

9. Ensemble Output & Class Imbalance: Final predictions sum all log-odds $z = z_0 + \eta \sum h_m(x)$, converted to probability via $p = \frac{1}{1 + e^{-z}}$. Parameter `scale_pos_weight` ($w_{\text{pos}}$) multiplies $g_i$ and $h_i$ for positive samples ($y_i=1$), forcing the trees to prioritize minority class errors.

---


#### Concept Note : XGBoost
0. Setup :  (government_impersonation_scam vs romance_scam), one feature: mentions_IRS (0/1). Two training docs: doc1 (mentions_IRS=1, true label=1=gov_impersonation), doc2 (mentions_IRS=0, true label=0=romance).

1. Staring Prediction : Every example starts with same baseliine score, F_0 = 0 (log-odds), which through sigmoid mead p = 0.5. F0 (doc1) =  -> p1 sigmoid(0) = 0.5

2. How wrong, which direction as well how confident each correction should be ? g = p-y, h = p(1-p)

3. Find Best Split - Gain = ½ [ GL²/(HL+λ) + GR²/(HR+λ) − (GL+GR)²/(HL+HR+λ) ] − γ, where (G, H = summed gradients/hessians in each side; λ=L2 regularization, default 1; γ=min-gain-to-split, default 0)

4. Leaf Output Values = each leaf gets one number — how much to adjust predictions for examples that land there. w_leaf = −G_leaf / (H_leaf + λ). w_left  (doc2, mentions_IRS=0) = −0.5 / 1.25 = −0.4, w_right (doc1, mentions_IRS=1) = −(−0.5) / 1.25 = 0.4

5. Update preduction with learning rate, F₁ = F₀ + η·w_leaf (η = 0.3, XGBoost's default learning_rate). doc1: F1 = 0 + 0.3(0.4)  =  0.12 → p1 = sigmoid(0.12)  ≈ 0.530
doc2: F1 = 0 + 0.3(-0.4) = -0.12 → p2 = sigmoid(-0.12) ≈ 0.470. Doc1 (true=1) moved 0.5 → 0.530 (correct direction). Doc2 (true=0) moved 0.5 → 0.470 (correct direction)

6. Repeat - Recompyte using new predcitions, build another tree (possibly splitting on a differen feature this time)

```
F = np.zeros(n_samples)              # start at log-odds 0
for round in range(n_estimators):
    p = sigmoid(F)
    g = p - y
    h = p * (1 - p)
    tree = fit_tree_to_gradients(X, g, h, lam=1, gamma=0)   # finds best splits via Gain formula
    F = F + learning_rate * tree.predict(X)                 # add this tree's scaled output
```

7. XGBoost trains 10 parallel boosted sequences, one per typology, each producing its own score per round; softmax across the 10 final scores gives probabilities, same as logreg's last step. 
    - F_gov(doc1)        = 0 + 0.3(0.4)  =  0.12
    - F_family(doc1)      = 0 + 0.3(-0.2) = -0.06
    - F_investment(doc1)  = 0 + 0.3(-0.2) = -0.06
    - exp(0.12)=1.127, exp(-0.06)=0.942, exp(-0.06)=0.942, sum=3.011, p = [0.374, 0.313, 0.313]

8. Round 2 repeats this exact process — new gradients computed from the updated p, 3 new trees built (one per class again), each class's F score extended further. After all rounds finish, the 3 final F scores go through one last softmax to get final probabilities 

9. So in general: at every node, XGBoost tries every feature (and every threshold, for non-binary features) as a candidate split, computes Gain for each, keeps the best — and recurses into each resulting branch to check whether splitting again still helps, up to a max_depth limit (default 6)

10. Hyperparameters (bias/variance):
    - n_estimators (# boosting rounds): more -> ↓bias, ↑variance
    - max_depth (how deep each tree splits): deeper -> ↓bias, ↑variance
    - learning_rate/η: lower -> ↓variance, needs more n_estimators to compensate
    - lambda/λ (L2 on leaf weights, same λ as w_leaf formula above): higher -> ↓variance, slightly ↑bias
    - gamma/γ (min gain to split, same γ as Gain formula above): higher -> ↓variance, ↑bias
    - subsample (fraction of rows per tree): lower -> ↓variance (trees see different data)
    - colsample_bytree (fraction of features per tree): lower -> ↓variance (decorrelates trees)
    - min_child_weight (min hessian sum to allow a split): higher -> ↓variance, ↑bias
    Underfitting: too few n_estimators, too shallow max_depth, λ/γ/min_child_weight too high
    Overfitting: too many n_estimators (no early stopping), too deep, λ/γ too low, subsample/colsample near 1.0

11. L1 (alpha) vs L2 (lambda) on leaf weight: L2 just adds to the denominator
    (w=-G/(H+λ), shrinks proportionally, never exactly 0); L1 soft-thresholds G
    toward 0 first (can zero out a leaf's weight entirely — sparser trees)

12. Why 2nd-order (Hessian), not just gradient like classic GBM: Taylor-expands
    the loss to 2nd order for a better local approximation -> faster convergence,
    and it's what makes the closed-form w_leaf/Gain formulas above possible
    (plain GBM needs a numeric line search instead). Also: XGBoost builds trees
    level-wise + supports parallel split-finding; classic GBM is more sequential.

13. Missing values: XGBoost doesn't need imputation — it learns a default split
    direction for missing values at each node during training (same mechanism
    that made enable_categorical's unseen-category handling work in the code below)

14. Not universally "best": on this project's sparse TF-IDF (2000 mostly-zero
    columns), logreg (0.96) beat XGBoost (0.93) — trees gain little from feature
    interactions that mostly aren't there in bag-of-words text. XGBoost's edge
    is structured/tabular features with real interactions (the 5 LLM-extracted
    fields), not sparse linear text signal.

### Example: 
Classification Worked Example ($N=4$, $x=[1,2,8,9]$, $y=[0,0,1,1]$, $\lambda=0, \eta=0.3, \gamma=0$, initial log-odds $z_0=0 \implies p_0=0.5$):
* **Round 1:**
* $g_i = p_0 - y_i \implies g = [0.5, 0.5, -0.5, -0.5]$; $h_i = 0.5(1-0.5) = 0.25 \implies h = [0.25, 0.25, 0.25, 0.25]$.
* Split $x \le 5 \implies$ Left ($x \le 2$): $G_L = 1.0, H_L = 0.5$; Right ($x \ge 8$): $G_R = -1.0, H_R = 0.5$.
* Leaf weights: $w_L^* = -\frac{1.0}{0.5} = -2.0$; $w_R^* = -\frac{-1.0}{0.5} = +2.0$.
* $\text{Gain} = \frac{1}{2}\left[\frac{1^2}{0.5} + \frac{(-1)^2}{0.5} - 0\right] = 2.0 > 0$.
* Updates: $z_1 = z_0 + 0.3(w^*) \implies z_L = -0.6, z_R = +0.6 \implies p = [0.354, 0.354, 0.646, 0.646]$.

* **Round 2:**
* $g = [0.354, 0.354, -0.354, -0.354]$; $h = 0.354(0.646) = 0.229 \implies h = [0.229, 0.229, 0.229, 0.229]$.
* Split $x \le 5 \implies G_L = 0.708, H_L = 0.458 \implies w_L^* = -\frac{0.708}{0.458} = -1.546$; $w_R^* = +1.546$.
* $\text{Gain} = \frac{1}{2}\left[\frac{0.708^2}{0.458} + \frac{(-0.708)^2}{0.458} - 0\right] = 1.094 > 0$.
* Updates: $z_2 = z_1 + 0.3(w^*) \implies z_L = -1.064, z_R = +1.064 \implies p = [0.257, 0.257, 0.743, 0.743]$.

In [1]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y, p, eps=1e-15):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def gradient(y, p):    # g = p - y
    return p - y

def hessian(p):         # h = p(1-p)
    return p * (1 - p)

print("g:", gradient(np.array([1.0]), np.array([0.8])), "h:", hessian(np.array([0.8])))  # -0.2, 0.16

from sklearn.metrics import log_loss as sk_log_loss
y, z = np.array([1, 0, 1, 1, 0]), np.array([0.5, -1.2, 2.0, 0.1, -0.3])
p = sigmoid(z)
print("log loss (ours vs sklearn):", log_loss(y, p), sk_log_loss(y, p))

g: [-0.2] h: [0.16]
log loss (ours vs sklearn): 0.4126078734206417 0.4126078734206417


In [2]:
def leaf_score(g, h, lam=0.0):    # G^2 / (H + lambda)
    return (np.sum(g) ** 2) / (np.sum(h) + lam)

def leaf_value(g, h, lam=0.0):    # -G / (H + lambda)
    return -np.sum(g) / (np.sum(h) + lam)

print("leaf_score:", leaf_score(np.array([-0.2, -0.3]), np.array([0.16, 0.21])))  # ~0.676

leaf_score: 0.6756756756756757


In [ ]:
def best_split_xgb(X, g, h, lam=0.0):
    best_gain = -float("inf")
    best_feature, best_threshold = None, None
    parent_score = leaf_score(g, h, lam)

    for feature_idx in range(X.shape[1]):
        values = np.unique(X[:, feature_idx])
        thresholds = (values[:-1] + values[1:]) / 2

        for t in thresholds:
            left_mask = X[:, feature_idx] <= t
            g_left, h_left = g[left_mask], h[left_mask]
            g_right, h_right = g[~left_mask], h[~left_mask]

            if len(g_left) == 0 or len(g_right) == 0:
                continue

            gain = leaf_score(g_left, h_left, lam) + leaf_score(g_right, h_right, lam) - parent_score

            if gain > best_gain:
                best_gain = gain
                best_feature, best_threshold = feature_idx, t

    return best_feature, best_threshold, best_gain

In [ ]:
X = np.array([1, 2, 8, 9], dtype=float).reshape(-1, 1)
y_toy = np.array([0, 0, 1, 1], dtype=float)

def boosting_round(X, y, z, lam=0.0, lr=0.3):
    p = sigmoid(z)
    g, h = gradient(y, p), hessian(p)
    feature, threshold, gain = best_split_xgb(X, g, h, lam)
    left_mask = X[:, feature] <= threshold
    left_value = leaf_value(g[left_mask], h[left_mask], lam)
    right_value = leaf_value(g[~left_mask], h[~left_mask], lam)
    tree_output = np.where(left_mask, left_value, right_value)
    return z + lr * tree_output, gain, left_value, right_value

z = np.zeros(4)
for round_num in range(1, 3):
    z, gain, left_val, right_val = boosting_round(X, y_toy, z)
    print(f"round {round_num}: gain={gain:.3f}, values=({left_val:.3f}, {right_val:.3f}), p={np.round(sigmoid(z), 3)}")
# expect: round1 gain=4.0, p=[.354,.354,.646,.646]; round2 gain~2.19, p=[.257,.257,.743,.743]

In [ ]:
from xgboost import XGBClassifier

# approximate check against the real library -- internals differ in minor ways, but
# direction and scale should agree with the worked example above
xgb_toy = XGBClassifier(n_estimators=2, max_depth=1, learning_rate=0.3, reg_lambda=0,
                          base_score=0.5, eval_metric="logloss")
xgb_toy.fit(X, y_toy)
print("xgboost p:", np.round(xgb_toy.predict_proba(X)[:, 1], 3))

In [ ]:
class TreeNode:
    def __init__(self, g, h, lam):
        self.value = leaf_value(g, h, lam)   # computed up front so pruning can collapse back to this
        self.is_leaf = True
        self.feature = self.threshold = self.gain = None
        self.left = self.right = None

def grow_tree(X, g, h, lam=1.0, max_depth=2, depth=0):
    node = TreeNode(g, h, lam)
    if depth >= max_depth or len(g) < 2:
        return node   # only stopping rule during growth is depth/size -- NOT gain
    feature, threshold, gain = best_split_xgb(X, g, h, lam)
    if feature is None:
        return node
    left_mask = X[:, feature] <= threshold
    node.is_leaf = False
    node.feature, node.threshold, node.gain = feature, threshold, gain
    node.left = grow_tree(X[left_mask], g[left_mask], h[left_mask], lam, max_depth, depth + 1)
    node.right = grow_tree(X[~left_mask], g[~left_mask], h[~left_mask], lam, max_depth, depth + 1)
    return node

def prune_tree(node, gamma):
    if node.is_leaf:
        return
    prune_tree(node.left, gamma)    # deepest splits resolved first -- bottom-up
    prune_tree(node.right, gamma)
    if node.left.is_leaf and node.right.is_leaf and (node.gain - gamma) < 0:
        node.is_leaf = True         # collapse back; node.value already computed at construction
        node.left = node.right = None

def predict_tree(node, x_row):
    if node.is_leaf:
        return node.value
    branch = node.left if x_row[node.feature] <= node.threshold else node.right
    return predict_tree(branch, x_row)


z0 = np.zeros(4)
g0, h0 = gradient(y_toy, sigmoid(z0)), hessian(sigmoid(z0))
tree = grow_tree(X, g0, h0, lam=1.0, max_depth=2)   # lam=1 here (not 0) to see regularization act
print(f"root gain={tree.gain:.3f} (expect ~1.333); depth-1 gains={tree.left.gain:.3f},{tree.right.gain:.3f} (expect ~-0.267)")

prune_tree(tree, gamma=0.1)
print(f"root kept: {not tree.is_leaf} (gain-gamma={tree.gain-0.1:.3f} >= 0)")
print(f"leaf values after pruning: {tree.left.value:.3f}, {tree.right.value:.3f} (expect -0.667, +0.667)")

# Cover check: root Sigma h = 1.0 (4 pts at p=0.5, h=0.25 each). min_child_weight>0.5 would
# block the depth-1 split -- each resulting child's cover is exactly 0.5
print("cover(root):", np.sum(h0), "| cover(left child):", np.sum(h0[X[:,0]<=5]))

## Boosting: LightGBM

#### 0. Core idea

Also gradient boosting. Optimized for speed and scale via 3 changes from XGBoost: leaf-wise growth, histogram binning, GOSS sampling. Plus EFB for sparse features.

Requires: `uv add lightgbm` if not installed.


#### 1. Leaf-wise vs level-wise growth

XGBoost (level-wise): fills every leaf at the current depth before going deeper.
LightGBM (leaf-wise): always splits whichever single leaf has the highest gain, regardless of depth.

Setup: one split done already. Two leaves, each with a further-split gain on offer:
```
leaf_right: gain=0.9 if split again
leaf_left:  gain=0.1 if split again
```
Budget: 2 more splits.

Level-wise: splits BOTH leaves once each, even though leaf_left barely helps. Total gain captured = 0.9+0.1 = 1.0. Result: 4 leaves, symmetric, all at depth 2.

Leaf-wise: splits leaf_right first (0.9, highest). That creates a child leaf with a new further-split gain, say 0.5. Compare 0.5 against leaf_left's still-unsplit 0.1: 0.5 wins, so leaf_right's child gets split again, leaf_left is never touched. Total gain captured = 0.9+0.5 = 1.4. More gain from the same 2-split budget, but an asymmetric tree, leaf_left stays shallow, leaf_right's branch goes deep.

This asymmetry is exactly why LightGBM risks overfitting more easily on small data without capping num_leaves.


In [ ]:
import heapq

# simulate the "pick highest-gain leaf" decision leaf-wise growth makes
candidate_splits = [
    ("leaf_right", 0.9),
    ("leaf_left", 0.1),
]

# level-wise: split every candidate at this depth regardless of gain
level_wise_total_gain = sum(gain for _, gain in candidate_splits)
print("level-wise total gain (splits both):", level_wise_total_gain)

# leaf-wise: always pop the highest-gain leaf, re-add its children as new candidates
heap = [(-gain, name) for name, gain in candidate_splits]
heapq.heapify(heap)

leaf_wise_total_gain = 0
budget = 2
new_child_gain = 0.5  # illustrative: leaf_right's child, once split, offers this much

for _ in range(budget):
    neg_gain, name = heapq.heappop(heap)
    gain = -neg_gain
    leaf_wise_total_gain += gain
    print(f"splitting {name}, gain={gain}")
    if name == "leaf_right":
        heapq.heappush(heap, (-new_child_gain, "leaf_right_child"))

print("leaf-wise total gain:", leaf_wise_total_gain)

#### 2. Histogram-based splits

Bucket continuous features into a fixed number of bins upfront. Search for the best split among bins, not every unique raw value.

Worked example, amount_lost values [500, 2500, 10000, 40000]:
```
raw thresholds to check: 3 (between each pair of unique sorted values)
binned into 2 bins at boundary 5000: bin1={500,2500}, bin2={10000,40000}
thresholds to check now: 1 (the single bin boundary)
```
Far fewer candidates per node, at the cost of slightly coarser split granularity. This is the idea XGBoost's "hist" tree_method borrowed.


In [ ]:
import numpy as np

values = np.array([500, 2500, 10000, 40000])
raw_thresholds = (np.sort(values)[:-1] + np.sort(values)[1:]) / 2
print("raw candidate thresholds:", raw_thresholds)

bin_boundary = 5000
binned = np.where(values < bin_boundary, 0, 1)
print("binned assignment:", binned)
print("candidate thresholds after binning: 1 (just the bin boundary)")

#### 3. GOSS (Gradient-based One-Side Sampling)

Keep all large-gradient examples (most wrong, most informative). Randomly sample a fraction of small-gradient examples (already well-predicted, less informative). Reweight the sampled ones to keep the gradient sum unbiased.

Worked example, 10 gradients: [0.9, 0.88, 0.85, 0.1, 0.08, 0.07, 0.05, 0.04, 0.03, 0.02]
```
top 20% by |gradient| kept in full: [0.9, 0.88]  (2 examples)
sample 20% of remaining 8: say [0.1, 0.08]  (2 examples)
reweight factor: (1-0.2)/0.2 = 4x, applied to the sampled small-gradient examples
total used this round: 4 of 10 examples' full computation
```
Real reduction in per-round work. The reweighting is what keeps the estimate statistically unbiased despite dropping most of the small-gradient examples.

#### 4. EFB (Exclusive Feature Bundling)

Bundles sparse, mutually-exclusive features (rarely nonzero at the same time) into one combined feature. Exactly what TF-IDF's 2000 mostly-zero word columns look like, or one-hot categorical columns. Cuts effective dimensionality without losing information, since the bundled features almost never fire together anyway.

#### 5. Practical notes

Typically the fastest of this whole comparison set on large or sparse data. Main risk: overfitting on small datasets if num_leaves isn't capped, since leaf-wise growth can build very deep, lopsided trees fast.


In [ ]:
import numpy as np

gradients = np.array([0.9, 0.88, 0.85, 0.1, 0.08, 0.07, 0.05, 0.04, 0.03, 0.02])
top_frac, sample_frac = 0.2, 0.2

sorted_idx = np.argsort(-np.abs(gradients))
n_top = int(len(gradients) * top_frac)
top_idx = sorted_idx[:n_top]
rest_idx = sorted_idx[n_top:]

n_sample = int(len(rest_idx) * sample_frac)
rng = np.random.default_rng(0)
sampled_idx = rng.choice(rest_idx, size=n_sample, replace=False)

reweight = (1 - top_frac) / sample_frac
print("kept in full:", gradients[top_idx])
print("sampled (reweighted x{:.0f}):".format(reweight), gradients[sampled_idx])
print("total examples used:", n_top + n_sample, "of", len(gradients))

In [ ]:
from lightgbm import LGBMClassifier
import numpy as np

X = np.array([[1, 1], [1, 0], [0, 1], [0, 0]])  # mentions_IRS, urgency_language
y = ["gov", "gov", "romance", "romance"]

lgbm = LGBMClassifier(n_estimators=10, num_leaves=4, random_state=42, verbose=-1)
lgbm.fit(X, y)

print("predictions:", lgbm.predict(X))
print("predicted probabilities:\n", lgbm.predict_proba(X))

## Boosting: CatBoost

#### 0. Core idea

Also gradient boosting, built specifically around handling categorical features well. Two innovations: Ordered Target Statistics (for encoding categories) and Ordered Boosting (applies the same idea to the boosting loop itself).

Requires: `uv add catboost` if not installed.


#### 1. The leakage problem with naive target-mean encoding

Setup: 4 rows in a random permutation order, all with impersonated_entity="IRS agent" except row_b:
```
row_a: IRS agent, y=1  (1st in permutation)
row_b: tech support, y=0  (different category, ignored below)
row_c: IRS agent, y=1  (2nd occurrence of "IRS agent")
row_d: IRS agent, y=0  (3rd occurrence of "IRS agent")
```
Naive target-mean encoding replaces "IRS agent" everywhere with the average y across ALL its occurrences, computed once:
```
mean(1, 1, 0) = 0.667, applied identically to row_a, row_c, row_d
```
Row_a's own label (y=1) was used to compute row_a's own encoded feature value. The model gets to see a value partly built from the answer it is supposed to predict.

Worse case: if "IRS agent" appeared just once (only row_a), naive encoding gives row_a exactly y=1.0 as its feature value. That is the label, handed straight to the model.


In [ ]:
rows = [
    {"name": "row_a", "category": "IRS agent", "y": 1},
    {"name": "row_b", "category": "tech support", "y": 0},
    {"name": "row_c", "category": "IRS agent", "y": 1},
    {"name": "row_d", "category": "IRS agent", "y": 0},
]

irs_rows = [r for r in rows if r["category"] == "IRS agent"]
naive_encoding = sum(r["y"] for r in irs_rows) / len(irs_rows)

print("naive target-mean encoding for 'IRS agent':", naive_encoding)
print("applied identically to:", [r["name"] for r in irs_rows])
print("row_a's own label was used to build row_a's own feature value: leakage")

#### 2. Ordered Target Statistics: the fix

Encode each row using only PRIOR occurrences in the permutation order, never its own label.
```
row_a (1st occurrence): no prior "IRS agent" rows exist yet -> fall back to a
  global prior (e.g. overall mean y=0.5) -> encoded = 0.5, uninfluenced by
  row_a's own y=1

row_c (2nd occurrence): prior occurrences = [row_a: y=1] -> encoded = 1.0
  (average of PRIOR rows only, excludes row_c's own y=1)

row_d (3rd occurrence): prior occurrences = [row_a: y=1, row_c: y=1] ->
  encoded = (1+1)/2 = 1.0 (excludes row_d's own y=0 entirely, notice row_d's
  encoded value does not reflect its own true label at all)
```
Each row's feature value depends only on rows that came before it in the permutation. Never on itself.


In [ ]:
global_prior = 0.5

def ordered_target_stat(rows, target_category):
    encoded = {}
    prior_ys = []
    for r in rows:
        if r["category"] != target_category:
            continue
        if len(prior_ys) == 0:
            encoded[r["name"]] = global_prior
        else:
            encoded[r["name"]] = sum(prior_ys) / len(prior_ys)
        prior_ys.append(r["y"])  # this row's own y becomes available for LATER rows only
    return encoded

result = ordered_target_stat(rows, "IRS agent")
for name, value in result.items():
    print(f"{name}: encoded = {value}")

#### 3. Ordered Boosting

Same principle, applied to the boosting loop itself, not just encoding. Computes the gradient for a given row using a model that excluded that row (implemented efficiently through multiple permutations rather than literally retraining per row).

Fixes "prediction shift": standard gradient boosting (XGBoost, LightGBM, classic GBM all have this to some degree) implicitly overfits a little, because every tree's gradient computation used a model that had already seen that exact row's label during its own construction. Ordered boosting removes that circularity.

#### 4. Symmetric (oblivious) trees

Every node at a given depth uses the SAME split condition (same feature, same threshold) across the whole tree. Unlike XGBoost or LightGBM, where different branches at the same depth can split on completely different features (LightGBM's asymmetric leaf-wise tree from the LightGBM note is the opposite extreme).

More restrictive, less flexible fitting. Much faster at inference, acts as a built-in regularizer.

#### 5. Practical notes

No manual category-alignment step needed (compare to the set_categories dance XGBoost requires). Just declare which columns are categorical (cat_features=[...]) and pass raw strings directly, CatBoost runs Ordered TS internally.


In [ ]:
from catboost import CatBoostClassifier
import pandas as pd

X = pd.DataFrame({
    "impersonated_entity": ["IRS agent", "tech support", "IRS agent", "grandchild"],
    "urgency_language": [1, 0, 1, 1],
})
y = ["gov_impersonation", "tech_support_scam", "gov_impersonation", "family_emergency_scam"]

cb = CatBoostClassifier(iterations=20, verbose=False, cat_features=["impersonated_entity"], random_state=42)
cb.fit(X, y)

print("predictions:", cb.predict(X).flatten())
print("no manual encoding step was needed, raw strings passed directly")